# Clasificacion de Imagenes

decir que hay en la imagen, "Gallina Si"/ "Gallina NO"

## Se conecta al Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Definir la carpeta desde donde tiene que leer las imagenes

In [ ]:
dataset_path = '/content/drive/MyDrive/Dataset imagenes/PRUEBA - GALLINAS' # <--- ¡ACTUALIZA ESTA RUTA CON LA QUE COPIASTE DE TU DRIVE!

### Cómo encontrar la ruta correcta en Google Drive

1.  **Abre Google Drive** en tu navegador.
2.  **Navega a la carpeta** que contiene tu dataset (en este caso, `PRUEBA - GALLINAS`).
3.  **Haz clic derecho** en la carpeta `PRUEBA - GALLINAS`.
4.  Selecciona **'Obtener enlace'**.
5.  En la ventana que aparece, busca la **ruta de la carpeta**. No copies el enlace de compartir, sino la ruta que se muestra cuando estás navegando en Google Drive o, incluso mejor, utiliza la opción **'Copiar ruta'** si está disponible en la interfaz de Colab/Drive.

    *   Alternativamente, puedes navegar hasta la carpeta en la interfaz de archivos de Colab (el icono de la carpeta en el panel izquierdo), hacer clic derecho en la carpeta `PRUEBA - GALLINAS` dentro de `drive/MyDrive`, y seleccionar **'Copiar ruta'**.

Una vez que tengas la ruta correcta, actualiza la variable `dataset_path` en la celda anterior (`7iV3gJIF5gbH`).

**¡Importante!** Asegúrate de que la estructura de tu dataset dentro de esa carpeta sea la esperada por `image_dataset_from_directory`, como se sugiere en el mensaje de error:

```
Prueba_gallinas/
│
├── train/
│   ├── gallina-SI/
│   └── gallina-NO/
│
└── valid/
    ├── gallina-SI/
    └── gallina-NO/
```

## importa las librerias


In [ ]:
import tensorflow as tf
import os
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPooling2D

import matplotlib.pyplot as plt
import numpy as np

## Carga del dataset

### Datos de entrenamiento y testeo

In [ ]:


# Rutas específicas
train_path = os.path.join(dataset_path, "Train")
valid_path = os.path.join(dataset_path, "Valid")

# Verificar existencia
if os.path.exists(dataset_path):

    print(f"Contenido de la carpeta principal '{dataset_path}':")

    for item in os.listdir(dataset_path):
        print(f" - {item}")

    # Parámetros
    IMG_HEIGHT = 128
    IMG_WIDTH = 128
    BATCH_SIZE = 16

    try:

        # DATASET DE ENTRENAMIENTO
        train_ds = tf.keras.utils.image_dataset_from_directory(

            train_path,
            labels='inferred',
            label_mode='int',
            image_size=(IMG_HEIGHT, IMG_WIDTH),
            interpolation='nearest',
            batch_size=BATCH_SIZE,
            shuffle=True,
            seed=123
        )

        # DATASET DE VALIDACIÓN
        val_ds = tf.keras.utils.image_dataset_from_directory(

            valid_path,
            labels='inferred',
            label_mode='int',
            image_size=(IMG_HEIGHT, IMG_WIDTH),
            interpolation='nearest',
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        # Clases detectadas
        class_names = train_ds.class_names

        num_classes = len(class_names)

        print("\nClases encontradas:")
        print(class_names)

        print(f"\nNúmero de clases: {num_classes}")

        # Advertencia si solo hay una clase
        if num_classes == 1:

            print("\n⚠️ Advertencia:")
            print("Solo se detectó una clase.")

        # Optimización
        AUTOTUNE = tf.data.AUTOTUNE

        train_ds = train_ds.cache().shuffle(1000).prefetch(
            buffer_size=AUTOTUNE
        )

        val_ds = val_ds.cache().prefetch(
            buffer_size=AUTOTUNE
        )

        print("\n✅ Dataset cargado correctamente")

    except ValueError as e:

        print(f"\n❌ Error al cargar dataset:\n{e}")

        print("\nVerifica estructura:")

        print("""
Prueba_gallinas/
│
├── train/
│   ├── gallina-SI/
│   └── gallina-NO/
│
└── valid/
    ├── gallina-SI/
    └── gallina-NO/
""")

else:

    print(f"\n❌ La ruta no existe:\n{dataset_path}")


❌ La ruta no existe:
/content/drive/MyDrive/Dataset imagenes/PRUEBA - GALLINAS


### Validacion del contenido

(esta comentado, hay que revisar dado que demora demasiado)

In [ ]:
"""import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):  # Toma un lote de imágenes del dataset de entrenamiento
  for i in range(9): # Muestra las primeras 9 imágenes del lote
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")
plt.show()
"""

'import matplotlib.pyplot as plt\n\nplt.figure(figsize=(10, 10))\nfor images, labels in train_ds.take(1):  # Toma un lote de imágenes del dataset de entrenamiento\n  for i in range(9): # Muestra las primeras 9 imágenes del lote\n    ax = plt.subplot(3, 3, i + 1)\n    plt.imshow(images[i].numpy().astype("uint8"))\n    plt.title(class_names[labels[i]])\n    plt.axis("off")\nplt.show()\n'

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPooling2D
#from tensorflow.keras.datasets import cifar10
import matplotlib.pyplot as plt
import numpy as np



### Normaliza las imagenes

In [ ]:
def normalize_img(image, label):
    """Normalizes images: `uint8` -> `float32`."""
    return tf.cast(image, tf.float32) / 255.0, label

# Aplicar la normalización a los datasets
train_ds = train_ds.map(normalize_img)
val_ds = val_ds.map(normalize_img)

print("Datasets normalizados exitosamente.")

NameError: name 'train_ds' is not defined

## Creación del Modelo CNN



In [ ]:
modelo = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),

    Flatten(),
    Dropout(0.5), #se agrega dps de observar un leve sobreajuste
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')# FUNCION SIGMOIDE PARA CLASIFICASION BINARIA
])

modelo.summary()

## Compilación

In [ ]:
modelo.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## Entrenamiento

In [ ]:
historial=modelo.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10)


### Comparativa

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(historial.history["accuracy"], label="la del entrenamiento")
plt.plot(historial.history["val_accuracy"], label="la de validacion")
plt.xlabel("Epocas")
plt.ylabel("Precision")
plt.legend()
plt.show()

**Se observa que la linea de entrenamiento de mantiene alta, con algunos altivajos pero mejora en cada epoca, lo que es de esperar. En la linea naranja se mantiene cercana a la precision de entrenamiento, y se observa uns caida en la epoca 4, lo que sugiere alguna clase de dificultad, que se supera y sigue subiendo su precisión hasta llegar al 0.97. En conclusión: Se concidera que el modelo generaliza bien, incluyendo un dropout luego del flatting, y con un total de 10 epocas. **

In [ ]:
modelo.save('modelo_entrenado.h5')

## Validaciones
Para observar algunas validaciones, se selecciona un lote, para elegir una imagen del data sat y probar el modelo. Puedes elegir otra cambiando el indice de la imagen y su etiqueta.

In [ ]:
loss, accuracy = modelo.evaluate(val_ds)
print(f"Perdida en testeo: {loss}")
print(f"Precision en testeo: {accuracy}")

# Obtener un lote del conjunto de validación
for images, labels in val_ds.take(1):
    imagen = images[15].numpy()  # Tomar la imagen en el índice 15 del lote
    etiqueta_real = labels[15].numpy() # Tomar la etiqueta en el índice 15 del lote
    break # Salir después de tomar el primer lote

plt.imshow(imagen)
plt.show()


prediccion = modelo.predict(np.array([imagen]))
# Convertir la predicción de probabilidad a una etiqueta de clase (0 o 1)
etiqueta_predicha = 1 if prediccion[0][0] > 0.5 else 0
print(f"Etiqueta real: {class_names[etiqueta_real]}")
print(f"Etiqueta predicha: {class_names[etiqueta_predicha]}")

In [ ]:
print("Buscando una imagen de 'Gallina - SI' del lote de validación...")

found_chicken_si = False
for images, labels in val_ds:
    for i in range(len(labels)):
        if labels[i].numpy() == 1: # 'Gallina - SI' corresponde al label 1
            imagen_si = images[i].numpy()
            etiqueta_real_si = labels[i].numpy()
            found_chicken_si = True
            break
    if found_chicken_si:
        break

if found_chicken_si:
    plt.imshow(imagen_si)
    plt.title(f"Imagen real: {class_names[etiqueta_real_si]}")
    plt.axis("off")
    plt.show()

    # Realizar la predicción
    prediccion_si = modelo.predict(np.expand_dims(imagen_si, axis=0))
    etiqueta_predicha_si = 1 if prediccion_si[0][0] > 0.5 else 0

    print(f"Etiqueta real: {class_names[etiqueta_real_si]}")
    print(f"Etiqueta predicha: {class_names[etiqueta_predicha_si]}")
else:
    print("No se encontró ninguna imagen de 'Gallina - SI' en el dataset de validación.")

## Probar el modelo con una imagen de la web

Primero, cargaremos una imagen de ejemplo desde la libreria Skimage, se preprocesa para que coincida con el tamaño y formato que espera el modelo, y luego realiza una predicción.

In [ ]:
import skimage.data
from skimage.transform import resize

print("Cargando imagen de gato desde skimage.data...")

# Cargar una imagen  de ejemplo de skimage
# Hay varias imágenes disponibles, 'cat' es una de ellas, puedes buscar otra .
img = skimage.data.cat()

# Redimensionar la imagen al tamaño esperado por el modelo
# skimage.transform.resize normaliza los píxeles a [0, 1] por defecto
img_resized_skimage = resize(img, (IMG_HEIGHT, IMG_WIDTH), anti_aliasing=True)

# Añadir una dimensión de batch (el modelo espera [batch_size, height, width, channels])
img_for_prediction_skimage = np.expand_dims(img_resized_skimage, axis=0)

print("Imagen de librería cargada y preprocesada exitosamente.")

# Mostrar la imagen que se va a clasificar
plt.imshow(img_resized_skimage)
plt.title("Imagen de prueba (Skimage Cat)")
plt.axis("off")
plt.show()

# Realizar la predicción
prediction_raw_skimage = modelo.predict(img_for_prediction_skimage)
probabilidad_skimage = prediction_raw_skimage[0][0] # Para una salida sigmoide

# Determinar la etiqueta predicha para clasificación binaria
if probabilidad_skimage > 0.5:
    etiqueta_predicha_idx_skimage = 1
else:
    etiqueta_predicha_idx_skimage = 0

if num_classes > 0 and etiqueta_predicha_idx_skimage < len(class_names):
    etiqueta_predicha_nombre_skimage = class_names[etiqueta_predicha_idx_skimage]
else:
    etiqueta_predicha_nombre_skimage = f"Índice {etiqueta_predicha_idx_skimage} (solo se encontró {num_classes} clase)"

print(f"\nProbabilidad de ser la clase 1 (sigmoid): {probabilidad_skimage:.4f}")
print(f"Probabilidad de ser la clase 0: {1-probabilidad_skimage:.4f}")
print(f"Etiqueta predicha: {etiqueta_predicha_nombre_skimage}")
